# Deep Learning 基礎講座　最終課題: 脳波分類

## 概要
被験者が画像を見ているときの脳波から，その画像がどのカテゴリに属するかを分類するタスク．
- サンプル数: 訓練 118,800 サンプル，検証 59,400 サンプル，テスト 59,400 サンプル
- クラス数: 5
- 入力: 脳波データ（チャンネル数 x 系列長）
- 出力: 対応する画像のクラス
- 評価指標: Top-1 accuracy

### 元データセット ([Gifford2022 EEG dataset](https://osf.io/3jk45/)) との違い

- 本コンペでは難易度調整の目的で元データセットにいくつかの改変を加えています．

1. 訓練セットのみの使用
  - 元データセットでは訓練データに存在しなかったクラスの画像を見ているときの脳波においてテストが行われますが，これは難易度が非常に高くなります．
  - 本コンペでは元データセットの訓練セットを再分割し，訓練時に存在した画像に対応する別の脳波において検証・テストを行います．

2. クラス数の減少
  - 元データセット（の訓練セット）では16,540枚の画像に対し，1,654のクラスが存在します．
    - e.g. `aardvark`, `alligator`, `almond`, ...
  - 本コンペでは1,654のクラスを，`animal`, `food`, `clothing`, `tool`, `vehicle`の5つにまとめています．
    - e.g. `aardvark -> animal`, `alligator -> animal`, `almond -> food`, ...

### 考えられる工夫の例

- 音声モデルの導入
  - 脳波と同じ波である音声を扱うアーキテクチャを用いることが有効であると知られています．
  - 例）Conformer [[Gulati+ 2020](https://arxiv.org/abs/2005.08100)]
- 画像データを用いた事前学習
  - 本コンペのタスクは脳波のクラス分類ですが，配布してある画像データを脳波エンコーダの事前学習に用いることを許可します．
  - 例）CLIP [Radford+ 2021]
  - 画像を用いる場合は[こちら](https://osf.io/download/3v527/)からダウンロードしてください．
- 過学習を防ぐ正則化やドロップアウト


## 修了要件を満たす条件
- ベースラインモデルのbest test accuracyは38.8%となります．**これを超えた提出のみ，修了要件として認めます**．
- ベースラインから改善を加えることで，55%までは性能向上することを運営で確認しています．こちらを 1 つの指標として取り組んでみてください．

## 注意点
- 最終的な予測モデルは，**配布している訓練データを用いて学習**（ファインチューニング含む）したものとしてください．
- 学習を行わず，**事前学習済みモデルの知識のみを利用した推論は禁止**します．  
（例: ChatGPT 等の LLM に入力して推論を得るのみ）

### 事前学習モデルの利用
許可される事項
- **構成要素としての事前学習モデルの利用**: 自身で実装したアーキテクチャの一部（特徴抽出，埋め込みなど）として事前学習モデル（BERT，ViT など）を利用することは可能です．
- **ファインチューニング**: 上記の用途で利用している事前学習モデルのファインチューニングは可能です．

禁止される事項  
- **タスク解決用の事前学習モデルの利用**: transformers などで提供されている，対象タスクを直接解くための事前学習モデルでそのまま推論のみ，またはファインチューニングのみで利用することは禁止とします．
  - 禁止事項の例: VQA タスクを直接解くための事前学習モデルを VQA タスクで利用する．

## 1.準備

In [1]:
# omnicampus 実行用
!pip install ipywidgets


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
# ライブラリのインポートとシード固定
import os, sys
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter
from einops.layers.torch import Rearrange
from einops import repeat
from glob import glob
from termcolor import cprint
from tqdm.notebook import tqdm

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# For Colab

In [ ]:
# ドライブのマウント（Colabの場合）
from google.colab import drive
drive.mount('/content/drive')

# For Local

In [2]:
# Set the working directory
import os
import numpy as np
import pandas as pd

#work_dir = os.path.dirname(os.path.dirname(os.getcwd())) 
work_dir = os.path.dirname(os.getcwd())

print(f"Current working directory: {work_dir}")

Current working directory: c:\Users\dysk-\Desktop\Current task\EEG compe


In [3]:
# ワーキングディレクトリを作成し移動．ノートブックを配置したディレクトリに適宜書き換え
#WORK_DIR = "/content/drive/MyDrive/weblab/DLBasics2025/Competition"
WORK_DIR = os.path.join(work_dir)
os.makedirs(WORK_DIR, exist_ok=True)
%cd {WORK_DIR}

c:\Users\dysk-\Desktop\Current task\EEG compe


## 2.データセット

ノートブックと同じディレクトリに`data/`が存在することを確認してください．

In [ ]:
class ThingsEEGDataset(torch.utils.data.Dataset):
    def __init__(self, split: str) -> None:
        super().__init__()

        assert split in ["train", "val", "test"], f"Invalid split: {split}"
        self.split = split
        self.num_classes = 5
        self.num_subjects = 10

        self.X = np.load(f"data/{split}/eeg.npy")

        # trial-wise z-score
        self.X = (
            self.X - self.X.mean(axis=-1, keepdims=True)
        ) / (self.X.std(axis=-1, keepdims=True) + 1e-6)

        # clipping
        self.X = np.clip(self.X, -5, 5)

        if split == "train":
            print(
                "clipped X mean/std/min/max:",
                self.X.mean(), self.X.std(), self.X.min(), self.X.max()
            )


        if split == "train":
            print("clipped X mean/std/min/max:", self.X.mean(), self.X.std(), self.X.min(), self.X.max())

        self.X = torch.from_numpy(self.X).to(torch.float32)
        self.subject_idxs = np.load(f"data/{split}/subject_idxs.npy")
        self.subject_idxs = torch.from_numpy(self.subject_idxs)

        if split in ["train", "val"]:
            self.img_features = np.load(f"data/{split}/vit_features.npy")
            self.img_features = torch.from_numpy(self.img_features).to(torch.float32)

        print(f"EEG: {self.X.shape}, labels: {self.y.shape if hasattr(self, 'y') else None}, subject indices: {self.subject_idxs.shape}")

    def __len__(self) -> int:
        return len(self.X)

    def __getitem__(self, i):
        if hasattr(self, "y"):
            return self.X[i], self.img_features[i], self.y[i], self.subject_idxs[i]
        else:
            return self.X[i], self.subject_idxs[i]

    @property
    def num_channels(self) -> int:
        return self.X.shape[1]

    @property
    def seq_len(self) -> int:
        return self.X.shape[2]

# 2.5 Load Config file

In [56]:
del run_dir

In [5]:


from pathlib import Path
from datetime import datetime
import json
import shutil

# ===== 読み込むconfigを指定 =====
#CONFIG_PATH =  Path("configs/baseline.json")
#CONFIG_PATH =  Path("configs/clip_m5_5.json")
#CONFIG_PATH =  Path("configs/baseline_zscore_clip.json")
#CONFIG_PATH =  Path("configs/eegnet_zscore_clip.json")
CONFIG_PATH =  Path("configs/eegnet_zscore_clip_SubjectEmbedding.json")

print(f"Loading config from: {CONFIG_PATH}")
#CONFIG_PATH = work_dir + CONFIG_PATH

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = json.load(f)

# ===== configから変数に反映 =====
RUN_NAME = config["run_name"]
seed = config["seed"]
lr = config["lr"]
batch_size = config["batch_size"]
epochs = config["epochs"]
model_name = config["model_name"]
optimizer_name = config["optimizer"]
scheduler_name = config["scheduler"]

# ===== 保存先作成 =====
if "run_dir" not in globals():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    run_dir = Path("outputs") / f"{timestamp}_{RUN_NAME}"
    run_dir.mkdir(parents=True, exist_ok=True)

    shutil.copy(CONFIG_PATH, run_dir / "config.json")

print(f"Run directory: {run_dir}")

Loading config from: configs\eegnet_zscore_clip_SubjectEmbedding.json
Run directory: outputs\20260609_1515_eegnet_zscore_clip_SubjectEmbedding


# Load image_features data

In [6]:
from pathlib import Path
import numpy as np

feature_path = work_dir + "/data/features/vit_image_features.npy"
path_txt = work_dir + "/data/features/vit_image_paths.txt"
print(feature_path)


features = np.load(feature_path)

with open(path_txt) as f:
    feature_paths = [p.strip() for p in f.readlines()]

print(features.shape)
print(len(feature_paths))
print(feature_paths[0])

c:\Users\dysk-\Desktop\Current task\EEG compe/data/features/vit_image_features.npy
(5940, 768)
5940
00001_aardvark/aardvark_01b.jpg


In [8]:
# path -> feature の辞書
feature_dict = {
    p: feat
    for p, feat in zip(feature_paths, features)
}

def make_trial_image_features(split):
    path_file = work_dir + f"/data/{split}/image_paths.txt"

    with open(path_file) as f:
        trial_paths = [p.strip() for p in f.readlines()]

    trial_features = np.stack([
        feature_dict[p]
        for p in trial_paths
    ])

    return trial_features

train_img_feats = make_trial_image_features("train")
val_img_feats = make_trial_image_features("val")

print(train_img_feats.shape)
print(val_img_feats.shape)

(118800, 768)
(59400, 768)


In [9]:
np.save(work_dir + "/data/train/vit_features.npy", train_img_feats)
np.save(work_dir + "/data/val/vit_features.npy", val_img_feats)

## 3.ベースラインモデル

In [10]:
class ConvBlock(nn.Module):
    def __init__(
        self,
        in_dim,
        out_dim,
        kernel_size: int = 3,
        p_drop: float = 0.1,
    ) -> None:
        super().__init__()

        self.in_dim = in_dim
        self.out_dim = out_dim

        self.conv0 = nn.Conv1d(in_dim, out_dim, kernel_size, padding="same")
        self.conv1 = nn.Conv1d(out_dim, out_dim, kernel_size, padding="same")
        # self.conv2 = nn.Conv1d(out_dim, out_dim, kernel_size) # , padding="same")

        self.batchnorm0 = nn.BatchNorm1d(num_features=out_dim)
        self.batchnorm1 = nn.BatchNorm1d(num_features=out_dim)

        self.dropout = nn.Dropout(p_drop)

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        if self.in_dim == self.out_dim:
            X = self.conv0(X) + X  # skip connection
        else:
            X = self.conv0(X)

        X = F.gelu(self.batchnorm0(X))

        X = self.conv1(X) + X  # skip connection
        X = F.gelu(self.batchnorm1(X))

        # X = self.conv2(X)
        # X = F.glu(X, dim=-2)

        return self.dropout(X)


class BasicConvClassifier(nn.Module):
    def __init__(
        self,
        num_classes: int,
        seq_len: int,
        in_channels: int,
        hid_dim: int = 128
    ) -> None:
        super().__init__()

        self.blocks = nn.Sequential(
            ConvBlock(in_channels, hid_dim),
            ConvBlock(hid_dim, hid_dim),
        )

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            Rearrange("b d 1 -> b d"),
            nn.Linear(hid_dim, num_classes),
        )

    def forward(self, X: torch.Tensor) -> torch.Tensor:
        """_summary_
        Args:
            X ( b, c, t ): _description_
        Returns:
            X ( b, num_classes ): _description_
        """
        X = self.blocks(X)

        return self.head(X)
    


class EEGNetClassifier(nn.Module):
    def __init__(
        self,
        num_classes: int,
        num_channels: int,
        seq_len: int,
        F1: int = 32,
        D: int = 2,
        F2: int = 64,
        dropout: float = 0.5,
        subject_emb_dim: int = 16,
        num_subjects: int = 10,
    ):
        super().__init__()

        self.temporal = nn.Sequential(
            nn.Conv2d(1, F1, kernel_size=(1, 15), padding=(0, 7), bias=False),
            nn.BatchNorm2d(F1),
        )

        self.spatial = nn.Sequential(
            nn.Conv2d(F1, F1 * D, kernel_size=(num_channels, 1), groups=F1, bias=False),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(dropout),
        )

        self.separable = nn.Sequential(
            nn.Conv2d(F1 * D, F1 * D, kernel_size=(1, 15), padding=(0, 7),
                      groups=F1 * D, bias=False),
            nn.Conv2d(F1 * D, F2, kernel_size=(1, 1), bias=False),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d(kernel_size=(1, 4)),
            nn.Dropout(dropout),
        )

        self.subject_embedding = nn.Embedding(num_subjects, subject_emb_dim)

        with torch.no_grad():
            dummy = torch.zeros(1, num_channels, seq_len)
            feat = self._forward_features(dummy)
            feat_dim = feat.shape[1]

        self.classifier = nn.Linear(feat_dim + subject_emb_dim, num_classes)

    def _forward_features(self, x):
        x = x.unsqueeze(1)  # (batch, 1, channels, time)
        x = self.temporal(x)
        x = self.spatial(x)
        x = self.separable(x)
        x = x.flatten(start_dim=1)
        return x

    def forward(self, x, subject_idxs):
        x = self._forward_features(x)
        subject_emb = self.subject_embedding(subject_idxs)
        x = torch.cat([x, subject_emb], dim=1)
        return self.classifier(x)

# Set Seed

In [11]:
import random
import numpy as np
import torch

def seed_everything(seed=1234):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

seed_everything(seed)

## 4.訓練実行

In [12]:


# ------------------
#    Dataloader
# ------------------
train_set = ThingsEEGDataset("train") # ThingsMEGDataset("train")
train_loader = torch.utils.data.DataLoader(
    train_set, batch_size=batch_size, shuffle=True
)
val_set = ThingsEEGDataset("val") # ThingsMEGDataset("val")
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=batch_size, shuffle=False
)

# ------------------
#       Model
# ------------------
model = EEGNetClassifier(
    num_classes=train_set.num_classes,
    num_channels=train_set.num_channels,
    seq_len=train_set.seq_len,
    F1=32,
    D=2,
    F2=64,
    dropout=0.5,
    subject_emb_dim=16,
    num_subjects=10,
).to("cuda")

# ------------------
#     Optimizer
# ------------------
#optimizer = torch.optim.Adam(model.parameters(), lr=lr)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=epochs
)

# ------------------
#   Start training
# ------------------
max_val_acc = 0
def accuracy(y_pred, y):
    return (y_pred.argmax(dim=-1) == y).float().mean()

writer = SummaryWriter("tensorboard")

for epoch in range(epochs):
    print(f"Epoch {epoch+1}/{epochs}")

    train_loss, train_acc, val_loss, val_acc = [], [], [], []

    model.train()
    for X, img_feat, y, subject_idxs in tqdm(train_loader, desc="Train", leave=False):
        X, y = X.to("cuda"), y.to("cuda")

        subject_idxs = subject_idxs.to("cuda") - 1

        y_pred = model(X, subject_idxs)

        loss = F.cross_entropy(y_pred, y)
        train_loss.append(loss.item())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        acc = accuracy(y_pred, y)
        train_acc.append(acc.item())

    model.eval()
    for X, img_feat, y, subject_idxs in tqdm(val_loader, desc="Validation", leave=False):
        X = X.to("cuda")
        y = y.to("cuda")
        subject_idxs = subject_idxs.to("cuda") - 1

        with torch.no_grad():
            y_pred = model(X, subject_idxs)

        val_loss.append(F.cross_entropy(y_pred, y).item())
        val_acc.append(accuracy(y_pred, y).item())

    print(f"Epoch {epoch+1}/{epochs} | \
        train loss: {np.mean(train_loss):.3f} | \
        train acc: {np.mean(train_acc):.3f} | \
        val loss: {np.mean(val_loss):.3f} | \
        val acc: {np.mean(val_acc):.3f}")

    writer.add_scalar("train_loss", np.mean(train_loss), epoch)
    writer.add_scalar("train_acc", np.mean(train_acc), epoch)
    writer.add_scalar("val_loss", np.mean(val_loss), epoch)
    writer.add_scalar("val_acc", np.mean(val_acc), epoch)

    torch.save(model.state_dict(), f"{run_dir}/model_last.pt")

    if np.mean(val_acc) > max_val_acc:
        cprint("New best. Saving the model.", "cyan")
        torch.save(model.state_dict(), run_dir / "model_best.pt")
        max_val_acc = np.mean(val_acc)

scheduler.step()

clipped X mean/std/min/max: 1.7563702703936585e-06 0.9999800750817592 -5.0 5.0
clipped X mean/std/min/max: 1.7563702703936585e-06 0.9999800750817592 -5.0 5.0
EEG: torch.Size([118800, 17, 100]), labels: torch.Size([118800]), subject indices: torch.Size([118800])
EEG: torch.Size([59400, 17, 100]), labels: torch.Size([59400]), subject indices: torch.Size([59400])
Epoch 1/80


Train:   0%|          | 0/233 [00:00<?, ?it/s]

ValueError: not enough values to unpack (expected 4, got 3)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir tensorboard

## 5.評価

In [13]:
# ------------------
#    Dataloader
# ------------------
test_set = ThingsEEGDataset("test")
test_loader = torch.utils.data.DataLoader(
    test_set, batch_size=batch_size, shuffle=False
)

# ------------------
#       Model
# ------------------
model = EEGNetClassifier(
    num_classes=train_set.num_classes,
    num_channels=train_set.num_channels,
    seq_len=train_set.seq_len,
    F1=32,
    D=2,
    F2=64,
    dropout=0.5,
    subject_emb_dim=16,
    num_subjects=10,
).to("cuda")
model.load_state_dict(torch.load(f"{run_dir}/model_best.pt", map_location="cuda"))

# ------------------
#  Start evaluation
# ------------------
probs = []
model.eval()

with torch.no_grad():
    for X, subject_idxs in tqdm(test_loader, desc="Evaluation", leave=False):
        X = X.to("cuda")
        subject_idxs = subject_idxs.to("cuda") - 1

        logits = model(X, subject_idxs)
        p = F.softmax(logits, dim=1)

        probs.append(p.cpu())

submission = torch.cat(probs, dim=0).numpy()

np.save(run_dir / "submission.npy", submission)

print("submission shape:", submission.shape)
print("pred label count:", np.bincount(submission.argmax(axis=1), minlength=5))
print(f"Saved to {run_dir / 'submission.npy'}")

EEG: torch.Size([59400, 17, 100]), labels: None, subject indices: torch.Size([59400])


C:\Users\dysk-\AppData\Local\Temp\ipykernel_888\3560819804.py:23: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(f"{run_dir}/model_best.pt", 

Evaluation:   0%|          | 0/117 [00:00<?, ?it/s]

submission shape: (59400, 5)
pred label count: [11845 39191  4540  3821     3]
Saved to outputs\20260608_1409_eegnet_zscore_clip_SubjectEmbedding\submission.npy


## 提出方法

以下の3点をzip化し，Omnicampusの「最終課題 (EEG)」から提出してください．

- `submission.npy`
- `model_last.pt`や`model_best.pt`など，テストに使用した重み（拡張子は`.pt`のみ）
- 本Colab Notebook

In [14]:
from zipfile import ZipFile
from datetime import datetime
from pathlib import Path

timestamp = datetime.now().strftime("%Y%m%d_%H%M")
zip_name = run_dir / f"{timestamp}_submission.zip"

submission_path = run_dir / "submission.npy"
model_path = run_dir / "model_best.pt"
notebook_path = Path(work_dir) / "notebooks" / "DL_Basic_2026_Spring_Competition_EEG_baseline.ipynb"

with ZipFile(zip_name, "w") as zf:
    zf.write(submission_path, arcname="submission.npy")
    zf.write(model_path, arcname="model_best.pt")
    zf.write(notebook_path, arcname="DL_Basic_2026_Spring_Competition_EEG_baseline.ipynb")

print(f"Created: {zip_name}")

with ZipFile(zip_name, "r") as zf:
    print(zf.namelist())

Created: outputs\20260608_1409_eegnet_zscore_clip_SubjectEmbedding\20260608_1423_submission.zip
['submission.npy', 'model_best.pt', 'DL_Basic_2026_Spring_Competition_EEG_baseline.ipynb']
